In [5]:
import os
import glob
from google.colab import drive

# 1. Garante que o Drive está montado
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

pasta_destino = '/content/drive/MyDrive/llm_data'
os.makedirs(pasta_destino, exist_ok=True)
caminho_final = os.path.join(pasta_destino, 'input.txt')

# 2. Clona o corpus com as atas em texto puro (direto do GitHub)
if not os.path.exists('corpus-atas-copom'):
    print("Baixando acervo de atas em texto puro...")
    !git clone https://github.com/ajdavidl/corpus-atas-copom.git

# 3. Lista e consolida todos os arquivos .txt
arquivos = sorted(glob.glob('corpus-atas-copom/atas/*.txt'))
print(f"Total de atas encontradas no acervo: {len(arquivos)}")

texto_consolidado = []

for idx, arq in enumerate(arquivos, 1):
    with open(arq, 'r', encoding='utf-8', errors='ignore') as f:
        conteudo = f.read().strip()
        if len(conteudo) > 500:
            nome_arquivo = os.path.basename(arq)
            texto_consolidado.append(f"\n\n====================\n{nome_arquivo}\n====================\n\n")
            texto_consolidado.append(conteudo)

texto_final = "".join(texto_consolidado)

# 4. Grava no Google Drive
if len(texto_final) > 10000:
    with open(caminho_final, "w", encoding="utf-8") as f:
        f.write(texto_final)
    tamanho_mb = len(texto_final.encode('utf-8')) / (1024 * 1024)
    print("\n" + "="*50)
    print(f"Sucesso! Arquivo gerado em: {caminho_final}")
    print(f"Volume total consolidado: {tamanho_mb:.2f} MB | {len(texto_final):,} caracteres")
    print(f"Total de atas incluídas: {len(arquivos)}")
    print("="*50)
else:
    print(f"Erro: volume insuficiente ({len(texto_final)} caracteres).")

Baixando acervo de atas em texto puro...
Cloning into 'corpus-atas-copom'...
remote: Enumerating objects: 1125, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 1125 (delta 80), reused 101 (delta 37), pack-reused 976 (from 1)
Receiving objects: 100% (1125/1125), 22.33 MiB | 18.15 MiB/s, done.
Resolving deltas: 100% (571/571), done.
Total de atas encontradas no acervo: 253

Sucesso! Arquivo gerado em: /content/drive/MyDrive/llm_data/input.txt
Volume total consolidado: 7.90 MB | 8,004,771 caracteres
Total de atas incluídas: 253


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

pasta_base = '/content/drive/MyDrive/llm_data'
arquivos = ['train.bin', 'val.bin', 'tokenizer/vocab.json', 'tokenizer/merges.txt']

print("Verificando arquivos no Drive montado:")
for arq in arquivos:
    caminho = os.path.join(pasta_base, arq)
    status = "OK" if os.path.exists(caminho) else "NÃO ENCONTRADO"
    print(f" - {arq:<25} -> {status}")

Verificando arquivos no Drive montado:
 - train.bin                 -> OK
 - val.bin                   -> OK
 - tokenizer/vocab.json      -> OK
 - tokenizer/merges.txt      -> OK


In [4]:
# ==============================================================================
# CADERNO DE SUPORTE: INGESTÃO E CONSOLIDAÇÃO MULTIDOMÍNIO
# ==============================================================================
!pip install -q pypdf

import io
import os
import re
import glob
import time
import requests
from pypdf import PdfReader
from google.colab import drive

# 1. Configuração do ambiente e pastas
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

pasta_base = '/content/drive/MyDrive/llm_data'
pasta_raw = os.path.join(pasta_base, 'raw')
os.makedirs(pasta_raw, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# ------------------------------------------------------------------------------
# FONTE 1: Atas do Copom (Migra o input.txt existente para a pasta raw)
# ------------------------------------------------------------------------------
caminho_copom = os.path.join(pasta_raw, 'input_copom.txt')
caminho_antigo = os.path.join(pasta_base, 'input.txt')

if os.path.exists(caminho_antigo) and not os.path.exists(caminho_copom):
    os.rename(caminho_antigo, caminho_copom)
    print(f"[Fonte 1] Atas do Copom organizadas em: {caminho_copom}")
elif os.path.exists(caminho_copom):
    print(f"[Fonte 1] Atas do Copom já presentes ({os.path.getsize(caminho_copom) / (1024*1024):.2f} MB).")

# ------------------------------------------------------------------------------
# FONTE 2: Relatório de Política Monetária (RPM/RI) - Trimestral
# ------------------------------------------------------------------------------
caminho_rpm = os.path.join(pasta_raw, 'input_rpm.txt')

if not os.path.exists(caminho_rpm):
    print("\n[Fonte 2] Extraindo Relatórios de Política Monetária (2022-2026)...")
    edicoes_rpm = []
    for ano in [2022, 2023, 2024, 2025, 2026]:
        for mes in ["03", "06", "09", "12"]:
            pref = "rpm" if ano >= 2025 else "ri"
            edicoes_rpm.append((f"{pref}{mes}/{ano}", f"https://www.bcb.gov.br/content/ri/relatorioinflacao/{ano}{mes}/{pref}{ano}{mes}p.pdf"))

    textos_rpm = []
    for rotulo, url in edicoes_rpm:
        try:
            r = requests.get(url, headers=headers, timeout=20)
            if r.status_code == 200 and len(r.content) > 50000:
                reader = PdfReader(io.BytesIO(r.content))
                txt = "\n".join([p.extract_text() or "" for p in reader.pages])
                txt = re.sub(r'Relatório de (Política Monetária|Inflação)\s*\|\s*.*', '', txt).strip()
                textos_rpm.append(f"\n\n<|FONTE: RELATORIO_POLITICA_MONETARIA | EDICAO: {rotulo}|>\n\n{txt}")
                print(f"  -> OK: {rotulo} ({len(reader.pages)} páginas)")
        except Exception:
            pass
        time.sleep(0.3)

    with open(caminho_rpm, "w", encoding="utf-8") as f:
        f.write("".join(textos_rpm))
    print(f"RPM salvo: {os.path.getsize(caminho_rpm) / (1024*1024):.2f} MB")
else:
    print(f"[Fonte 2] RPM já presente ({os.path.getsize(caminho_rpm) / (1024*1024):.2f} MB).")

# ------------------------------------------------------------------------------
# FONTE 3: Relatório de Estabilidade Financeira (REF) - Semestral
# ------------------------------------------------------------------------------
caminho_ref = os.path.join(pasta_raw, 'input_ref.txt')

if not os.path.exists(caminho_ref):
    print("\n[Fonte 3] Extraindo Relatórios de Estabilidade Financeira (2021-2026)...")
    edicoes_ref = []
    for ano in [2021, 2022, 2023, 2024, 2025, 2026]:
        for mes in ["05", "11"]:
            edicoes_ref.append((f"REF {mes}/{ano}", f"https://www.bcb.gov.br/content/publicacoes/ref/{ano}{mes}/ref{ano}{mes}p.pdf"))

    textos_ref = []
    for rotulo, url in edicoes_ref:
        try:
            r = requests.get(url, headers=headers, timeout=20)
            if r.status_code == 200 and len(r.content) > 50000:
                reader = PdfReader(io.BytesIO(r.content))
                txt = "\n".join([p.extract_text() or "" for p in reader.pages])
                txt = re.sub(r'Relatório de Estabilidade Financeira\s*\|\s*.*', '', txt).strip()
                textos_ref.append(f"\n\n<|FONTE: RELATORIO_ESTABILIDADE_FINANCEIRA | EDICAO: {rotulo}|>\n\n{txt}")
                print(f"  -> OK: {rotulo} ({len(reader.pages)} páginas)")
        except Exception:
            pass
        time.sleep(0.3)

    with open(caminho_ref, "w", encoding="utf-8") as f:
        f.write("".join(textos_ref))
    print(f"REF salvo: {os.path.getsize(caminho_ref) / (1024*1024):.2f} MB")
else:
    print(f"[Fonte 3] REF já presente ({os.path.getsize(caminho_ref) / (1024*1024):.2f} MB).")

# ------------------------------------------------------------------------------
# FONTE 4: Normativos SUSEP / CNSP (Atos Normativos de Provisões e Solvência)
# ------------------------------------------------------------------------------
caminho_susep = os.path.join(pasta_raw, 'input_susep.txt')

if not os.path.exists(caminho_susep):
    print("\n[Fonte 4] Coletando compilação de normativos regulatórios SUSEP/CNSP...")
    # Principais normas de capital e provisões (Resoluções CNSP 321, 416, 432 e Circulares)
    urls_susep = [
        ("Resolução CNSP 321/2015", "https://www2.susep.gov.br/safe/scripts/bnweb/bnmapi.exe?router=upload/25242"),
        ("Circular SUSEP 662/2022", "https://www2.susep.gov.br/safe/scripts/bnweb/bnmapi.exe?router=upload/25882"),
        ("Circular SUSEP 700/2024", "https://www2.susep.gov.br/safe/scripts/bnweb/bnmapi.exe?router=upload/28306")
    ]
    textos_susep = []
    for titulo_norma, url in urls_susep:
        try:
            r = requests.get(url, headers=headers, timeout=15)
            if r.status_code == 200:
                txt = re.sub(r'<[^>]+>', ' ', r.text)
                txt = re.sub(r'\s+', ' ', txt).strip()
                if len(txt) > 2000:
                    textos_susep.append(f"\n\n<|FONTE: NORMATIVO_SUSEP | TITULO: {titulo_norma}|>\n\n{txt}")
                    print(f"  -> OK: {titulo_norma} ({len(txt):,} caracteres)")
        except Exception:
            pass

    with open(caminho_susep, "w", encoding="utf-8") as f:
        f.write("".join(textos_susep))
    print(f"SUSEP salva: {os.path.getsize(caminho_susep) / (1024*1024):.2f} MB")
else:
    print(f"[Fonte 4] SUSEP já presente ({os.path.getsize(caminho_susep) / (1024*1024):.2f} MB).")

# ------------------------------------------------------------------------------
# 5. Consolidação Master com Tags
# ------------------------------------------------------------------------------
print("\n" + "="*50)
print("Consolidando o arquivo Master...")
caminho_master = os.path.join(pasta_base, 'input_master.txt')

with open(caminho_master, 'w', encoding='utf-8') as out:
    for arq in [caminho_copom, caminho_rpm, caminho_ref, caminho_susep]:
        if os.path.exists(arq):
            with open(arq, 'r', encoding='utf-8', errors='ignore') as f:
                out.write(f.read() + "\n\n")

total_mb = os.path.getsize(caminho_master) / (1024 * 1024)
print(f"input_master.txt gerado com sucesso!")
print(f"Volume total consolidado: {total_mb:.2f} MB")
print("="*50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 30.2 MB/s eta 0:00:00
[Fonte 1] Atas do Copom já presentes (7.90 MB).
[Fonte 2] RPM já presente (4.59 MB).
[Fonte 3] REF já presente (1.54 MB).
[Fonte 4] SUSEP já presente (0.24 MB).

Consolidando o arquivo Master...
input_master.txt gerado com sucesso!
Volume total consolidado: 14.27 MB


In [7]:
import io
import os
import re
import time
import requests
from pypdf import PdfReader
from google.colab import drive

# 1. Garante montagem do Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

pasta_base = '/content/drive/MyDrive/llm_data'
pasta_raw = os.path.join(pasta_base, 'raw')
caminho_ref = os.path.join(pasta_raw, 'input_ref.txt')

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 2. Padrão oficial de URLs do REF no Banco Central (Maio e Outubro/Novembro)
anos_ref = [2021, 2022, 2023, 2024, 2025, 2026]
meses_ref = ["05", "10", "11"]

print("Extraindo Relatórios de Estabilidade Financeira (REF - BCB)...")
textos_ref = []

for ano in anos_ref:
    for mes in meses_ref:
        rotulo = f"REF {mes}/{ano}"
        # Testa variações maiúscula e minúscula do padrão do BCB
        candidatos_url = [
            f"https://www.bcb.gov.br/content/publicacoes/ref/{ano}{mes}/RELESTAB{ano}{mes}-refPub.pdf",
            f"https://www.bcb.gov.br/content/publicacoes/ref/{ano}{mes}/relestab{ano}{mes}-refpub.pdf"
        ]

        for url in candidatos_url:
            try:
                r = requests.get(url, headers=headers, timeout=25)
                if r.status_code == 200 and len(r.content) > 50000:
                    reader = PdfReader(io.BytesIO(r.content))
                    paginas = [p.extract_text() or "" for p in reader.pages]
                    txt = "\n".join(paginas)
                    txt = re.sub(r'Relatório de Estabilidade Financeira\s*\|\s*.*', '', txt).strip()

                    if len(txt) > 10000:
                        bloco = f"\n\n========================================\n<|FONTE: RELATORIO_ESTABILIDADE_FINANCEIRA | EDICAO: {rotulo}|>\n========================================\n\n{txt}"
                        textos_ref.append(bloco)
                        print(f"  -> OK: {rotulo} ({len(reader.pages)} páginas | {len(txt):,} caracteres)")
                        break
            except Exception:
                pass
        time.sleep(0.3)

# 3. Salva o input_ref.txt
if textos_ref:
    with open(caminho_ref, "w", encoding="utf-8") as f:
        f.write("".join(textos_ref))
    tamanho_ref_mb = os.path.getsize(caminho_ref) / (1024 * 1024)
    print(f"\nREF salvo com sucesso: {tamanho_ref_mb:.2f} MB")
else:
    print("\nAviso: nenhuma edição do REF foi baixada.")

# 4. Atualiza o input_master.txt consolidando todas as 4 fontes
print("\nAtualizando o arquivo consolidado input_master.txt...")
caminho_copom = os.path.join(pasta_raw, 'input_copom.txt')
caminho_rpm = os.path.join(pasta_raw, 'input_rpm.txt')
caminho_susep = os.path.join(pasta_raw, 'input_susep.txt')
caminho_master = os.path.join(pasta_base, 'input_master.txt')

fontes = [caminho_copom, caminho_rpm, caminho_ref, caminho_susep]

with open(caminho_master, 'w', encoding='utf-8') as out:
    for f_path in fontes:
        if os.path.exists(f_path):
            with open(f_path, 'r', encoding='utf-8', errors='ignore') as f:
                out.write(f.read() + "\n\n")

total_mb = os.path.getsize(caminho_master) / (1024 * 1024)
print("=" * 50)
print(f"Arquivo Master atualizado em: {caminho_master}")
print(f"Volume total consolidado: {total_mb:.2f} MB")
print("=" * 50)

Extraindo Relatórios de Estabilidade Financeira (REF - BCB)...
  -> OK: REF 10/2021 (96 páginas | 256,272 caracteres)
  -> OK: REF 10/2022 (85 páginas | 232,818 caracteres)
  -> OK: REF 10/2023 (78 páginas | 216,591 caracteres)
  -> OK: REF 10/2024 (94 páginas | 243,841 caracteres)
  -> OK: REF 10/2025 (98 páginas | 284,990 caracteres)
  -> OK: REF 05/2026 (113 páginas | 323,421 caracteres)

REF salvo com sucesso: 1.54 MB

Atualizando o arquivo consolidado input_master.txt...
Arquivo Master atualizado em: /content/drive/MyDrive/llm_data/input_master.txt
Volume total consolidado: 14.27 MB


In [8]:
import os
import re
from collections import Counter
import unicodedata

caminho_master = '/content/drive/MyDrive/llm_data/input_master.txt'

if not os.path.exists(caminho_master):
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_master}")

print("Carregando o corpus master em memória...")
with open(caminho_master, 'r', encoding='utf-8', errors='ignore') as f:
    texto = f.read()

# ----------------------------------------------------
# 1. Métricas Gerais de Volume
# ----------------------------------------------------
tamanho_mb = os.path.getsize(caminho_master) / (1024 * 1024)
num_caracteres = len(texto)
linhas = texto.splitlines()
num_linhas = len(linhas)
linhas_nao_vazias = sum(1 for l in linhas if l.strip())

# Extração de palavras (alfanuméricas normalizadas)
palavras_todas = re.findall(r'\b[a-zA-ZáàâãéêíóôõúüçÁÀÂÃÉÊÍÓÔÕÚÜÇ0-9_\-]+\b', texto)
num_palavras = len(palavras_todas)
palavras_lower = [p.lower() for p in palavras_todas]
vocab_palavras = set(palavras_lower)
vocab_size = len(vocab_palavras)
lexical_diversity = (vocab_size / num_palavras) * 100 if num_palavras else 0

print("=" * 65)
print("1. MÉTRICAS GERAIS DO CORPUS")
print("=" * 65)
print(f"Tamanho do arquivo:        {tamanho_mb:.2f} MB")
print(f"Total de caracteres:       {num_caracteres:,}")
print(f"Total de linhas:           {num_linhas:,} ({linhas_nao_vazias:,} com conteúdo)")
print(f"Total de tokens/palavras:  {num_palavras:,}")
print(f"Vocabulário único:         {vocab_size:,} palavras distintas")
print(f"Diversidade Lexical (TTR): {lexical_diversity:.2f}% (relação tipos/tokens)")

# ----------------------------------------------------
# 2. Distribuição por Fonte (<|FONTE:...|>)
# ----------------------------------------------------
fontes_matches = list(re.finditer(r'<\|FONTE:\s*([A-Z_]+)\s*\|\s*EDICAO:\s*([^|>]+)\|>', texto))

print("\n" + "=" * 65)
print("2. DISTRIBUIÇÃO E COBERTURA POR FONTE")
print("=" * 65)
if fontes_matches:
    contagem_fontes = Counter()
    caracteres_por_fonte = Counter()

    for i, match in enumerate(fontes_matches):
        nome_fonte = match.group(1).strip()
        inicio = match.start()
        fim = fontes_matches[i+1].start() if i+1 < len(fontes_matches) else len(texto)
        tamanho_secao = fim - inicio

        contagem_fontes[nome_fonte] += 1
        caracteres_por_fonte[nome_fonte] += tamanho_secao

    print(f"{'Fonte':<32} | {'Edições/Blocos':<15} | {'Caracteres':<15} | {'% Volume'}")
    print("-" * 75)
    for fonte, count in contagem_fontes.most_common():
        chars = caracteres_por_fonte[fonte]
        pct = (chars / num_caracteres) * 100
        print(f"{fonte:<32} | {count:<15} | {chars:<15,} | {pct:>6.2f}%")
else:
    print("Nenhuma tag estruturada <|FONTE:...|> detectada no arquivo.")

# ----------------------------------------------------
# 3. Análise de Caracteres e Símbolos Especiais
# ----------------------------------------------------
caracteres_unicos = sorted(list(set(texto)))
letras = [c for c in caracteres_unicos if c.isalpha()]
digitos = [c for c in caracteres_unicos if c.isdigit()]
pontuacao_espacos = [c for c in caracteres_unicos if not c.isalnum()]

print("\n" + "=" * 65)
print("3. ANÁLISE DE CARACTERES (COMPOSIÇÃO DO ALFABETO)")
print("=" * 65)
print(f"Total de caracteres únicos: {len(caracteres_unicos)}")
print(f"Dígitos numéricos:          {' '.join(digitos)}")
print(f"Símbolos/Pontuação comuns:  {' '.join([c for c in pontuacao_espacos if c not in '\n\r\t'][:35])}")

# Verifica se existem caracteres fora do padrão Latin/Unicode esperado
caracteres_estranhos = [c for c in caracteres_unicos if unicodedata.category(c).startswith('C') and c not in '\n\r\t']
if caracteres_estranhos:
    print(f"Aviso: {len(caracteres_estranhos)} caracteres de controle/invisíveis detectados.")
else:
    print("Integridade de caracteres: Nenhum caractere de controle corrompido.")

# ----------------------------------------------------
# 4. Top Termos Mais Frequentes (Domínio Financeiro/Atuarial)
# ----------------------------------------------------
stopwords_pt = {
    'de', 'a', 'o', 'que', 'e', 'do', 'da', 'em', 'um', 'para', 'é', 'com', 'não', 'uma',
    'os', 'no', 'se', 'na', 'por', 'mais', 'as', 'dos', 'como', 'mas', 'foi', 'ao', 'ele',
    'das', 'tem', 'à', 'seu', 'sua', 'ou', 'ser', 'quando', 'muito', 'há', 'nos', 'já',
    'está', 'eu', 'também', 'só', 'pelo', 'pela', 'até', 'isso', 'ela', 'entre', 'era',
    'depois', 'sem', 'mesmo', 'aos', 'ter', 'seus', 'quem', 'nas', 'me', 'esse', 'eles',
    'estão', 'você', 'tinha', 'foram', 'essa', 'num', 'nem', 'suas', 'meu', 'às', 'minha',
    'têm', 'numa', 'pelos', 'elas', 'havia', 'seja', 'qual', 'será', 'nós', 'tenho', 'lhe',
    'deles', 'essas', 'esses', 'pelas', 'este', 'fosse', 'dele', 'tu', 'te', 'vocês', 'vos',
    'lhes', 'meus', 'minhas', 'teu', 'tua', 'teus', 'tuas', 'nosso', 'nossa', 'nossos',
    'nossas', 'dela', 'delas', 'esta', 'estes', 'estas', 'aquele', 'aquela', 'aqueles',
    'aquelas', 'isto', 'aquilo', 'estou', 'está', 'estamos', 'estão', 'estive', 'esteve',
    'estivemos', 'estiveram', 'estava', 'estávamos', 'estavam', 'sobre', 'ano', 'anos',
    'período', 'dia', 'mês', 'meses', 'onde', 'assim', 'ainda', 'sob', 'cada', 'bem'
}

palavras_filtradas = [w for w in palavras_lower if len(w) > 2 and w not in stopwords_pt and not w.isdigit()]
top_termos = Counter(palavras_filtradas).most_common(25)

print("\n" + "=" * 65)
print("4. TOP 25 TERMOS TÉCNICOS MAIS RELEVANTES")
print("=" * 65)
for i, (termo, freq) in enumerate(top_termos, 1):
    barra = "█" * int((freq / top_termos[0][1]) * 20)
    print(f"{i:>2}. {termo:<18} | {freq:>7,} ocorrências {barra}")

# ----------------------------------------------------
# 5. Amostras Qualitativas de Trechos do Corpus
# ----------------------------------------------------
print("\n" + "=" * 65)
print("5. AMOSTRAS DE TEXTO EXTRAÍDAS DO CORPUS")
print("=" * 65)
posicoes = [0.05, 0.40, 0.75, 0.95]
for p in posicoes:
    idx = int(len(texto) * p)
    trecho = texto[idx:idx+350].strip().replace("\n", " ")
    print(f"\n[Amostra em ~{int(p*100)}% do arquivo]:")
    print(f'"{trecho}..."')


Carregando o corpus master em memória...
1. MÉTRICAS GERAIS DO CORPUS
Tamanho do arquivo:        14.27 MB
Total de caracteres:       14,316,241
Total de linhas:           186,647 (169,150 com conteúdo)
Total de tokens/palavras:  2,344,027
Vocabulário único:         27,066 palavras distintas
Diversidade Lexical (TTR): 1.15% (relação tipos/tokens)

2. DISTRIBUIÇÃO E COBERTURA POR FONTE
Fonte                            | Edições/Blocos  | Caracteres      | % Volume
---------------------------------------------------------------------------
RELATORIO_POLITICA_MONETARIA     | 19              | 4,614,224       |  32.23%
RELATORIO_ESTABILIDADE_FINANCEIRA | 6               | 1,697,242       |  11.86%

3. ANÁLISE DE CARACTERES (COMPOSIÇÃO DO ALFABETO)
Total de caracteres únicos: 2242
Dígitos numéricos:          0 1 2 3 4 5 6 7 8 9 ² ³ ¹ ٠ ١ ٢ ٣ ٥ ٦ ٧ ٨ ٩ ۰ ۱ ۲ ۳ ۴ ۵ ۶ ۷ ۸ ۹ ߀ ߂ ߄ ߅ ߆ ߇ ߈ ߉ ᠔ ᧖ ⁰ ⁴ ⁵ ₁ ⒉ ꘧
Símbolos/Pontuação comuns:                            ! " # $ % & ' 

In [9]:
import os
import re
import unicodedata

caminho_master = '/content/drive/MyDrive/llm_data/input_master.txt'
caminho_clean = '/content/drive/MyDrive/llm_data/input_clean.txt'

print("Carregando o corpus master para higienização...")
with open(caminho_master, 'r', encoding='utf-8', errors='ignore') as f:
    texto = f.read()

tamanho_original = len(texto)
print(f"Caracteres originais: {tamanho_original:,}")

# 1. Correção de quebras de hifenização (ex: 'econo-\n mia' ou 'econo- mia' -> 'economia')
print("Corrigindo hifenizações de quebra de linha...")
texto = re.sub(r'([a-zA-ZáàâãéêíóôõúüçÁÀÂÃÉÊÍÓÔÕÚÜÇ]+)-\s*\n\s*([a-zA-ZáàâãéêíóôõúüçÁÀÂÃÉÊÍÓÔÕÚÜÇ]+)', r'\1\2', texto)
texto = re.sub(r'([a-zA-ZáàâãéêíóôõúüçÁÀÂÃÉÊÍÓÔÕÚÜÇ]+)-\s+([a-zA-ZáàâãéêíóôõúüçÁÀÂÃÉÊÍÓÔÕÚÜÇ]{2,})', r'\1\2', texto)

# 2. Normalização Unicode (NFKC: unifica formas compatíveis, ex: frações, acentos compostos)
print("Aplicando normalização Unicode NFKC...")
texto = unicodedata.normalize('NFKC', texto)

# 3. Remoção de caracteres de controle, invisíveis e glifos espúrios de diagramação de PDF
print("Filtrando caracteres de controle e glifos inválidos...")
# Mantém: letras latinas, acentuadas, números, pontuação, símbolos matemáticos/moedas e quebras de linha
caracteres_permitidos = re.compile(
    r'[^\n\r\t a-zA-Z0-9áàâãéêíóôõúüçÁÀÂÃÉÊÍÓÔÕÚÜÇ'
    r'.,;:!?\'"«»“”„…\(\)\[\]\{\}\/\\\|\-_+=<>%&§$#@*~°ºª]'
)
texto = caracteres_permitidos.sub('', texto)

# 4. Normalização de linhas vazias e espaços em branco excessivos
print("Normalizando espaçamento e quebras de linha...")
texto = re.sub(r'[ \t]+', ' ', texto)               # Múltiplos espaços/tabs -> espaço simples
texto = re.sub(r' *\n *', '\n', texto)              # Espaços antes/depois de quebra de linha
texto = re.sub(r'\n{3,}', '\n\n', texto)            # Mais de duas quebras -> quebra de parágrafo dupla
texto = texto.strip()

# 5. Salva o corpus limpo
with open(caminho_clean, 'w', encoding='utf-8') as f:
    f.write(texto)

# Métricas pós-limpeza
tamanho_limpo_mb = os.path.getsize(caminho_clean) / (1024 * 1024)
caracteres_finais = len(texto)
chars_unicos_finais = len(set(texto))

print("\n" + "=" * 60)
print("HIGIENIZAÇÃO CONCLUÍDA")
print("=" * 60)
print(f"Arquivo gerado:              {caminho_clean}")
print(f"Tamanho em disco:            {tamanho_limpo_mb:.2f} MB")
print(f"Caracteres antes vs. depois: {tamanho_original:,} -> {caracteres_finais:,}")
print(f"Caracteres únicos (alfabeto): 2.242 -> {chars_unicos_finais} caracteres limpos")
print("=" * 60)

Carregando o corpus master para higienização...
Caracteres originais: 14,316,241
Corrigindo hifenizações de quebra de linha...
Aplicando normalização Unicode NFKC...
Filtrando caracteres de controle e glifos inválidos...
Normalizando espaçamento e quebras de linha...

HIGIENIZAÇÃO CONCLUÍDA
Arquivo gerado:              /content/drive/MyDrive/llm_data/input_clean.txt
Tamanho em disco:            13.91 MB
Caracteres antes vs. depois: 14,316,241 -> 14,102,171
Caracteres únicos (alfabeto): 2.242 -> 125 caracteres limpos


In [10]:
import os
import numpy as np
from tokenizers import ByteLevelBPETokenizer

pasta_base = '/content/drive/MyDrive/llm_data'
caminho_clean = os.path.join(pasta_base, 'input_clean.txt')
pasta_tokenizer = os.path.join(pasta_base, 'tokenizer')
os.makedirs(pasta_tokenizer, exist_ok=True)

# ----------------------------------------------------
# 1. Treinamento do Tokenizador BPE
# ----------------------------------------------------
vocab_size = 4096
special_tokens = ["<|endoftext|>", "<|pad|>", "<|unk|>"]

print(f"Treinando tokenizador ByteLevel BPE com vocab_size={vocab_size}...")
tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=[caminho_clean],
    vocab_size=vocab_size,
    min_frequency=2,
    special_tokens=special_tokens
)

# Salva a definição do tokenizador no Google Drive
tokenizer.save_model(pasta_tokenizer)
tokenizer.save(os.path.join(pasta_tokenizer, "tokenizer.json"))
print(f"Tokenizador salvo com sucesso em: {pasta_tokenizer}")

# ----------------------------------------------------
# 2. Codificação (Tokenização) do Corpus Master
# ----------------------------------------------------
print("\nCodificando o corpus completo em tokens...")
with open(caminho_clean, 'r', encoding='utf-8') as f:
    texto = f.read()

encoded = tokenizer.encode(texto)
tokens = np.array(encoded.ids, dtype=np.uint16)

num_tokens = len(tokens)
num_chars = len(texto)
taxa_compressao = num_chars / num_tokens if num_tokens else 0

print("=" * 60)
print(f"Total de caracteres de entrada: {num_chars:,}")
print(f"Total de tokens gerados:        {num_tokens:,}")
print(f"Taxa de compressão média:       {taxa_compressao:.2f} caracteres/token")
print("=" * 60)

# ----------------------------------------------------
# 3. Divisão Treino / Validação (90% / 10%) e Gravação Binária
# ----------------------------------------------------
split_idx = int(num_tokens * 0.9)
train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

caminho_train = os.path.join(pasta_base, 'train.bin')
caminho_val = os.path.join(pasta_base, 'val.bin')

train_tokens.tofile(caminho_train)
val_tokens.tofile(caminho_val)

train_mb = os.path.getsize(caminho_train) / (1024 * 1024)
val_mb = os.path.getsize(caminho_val) / (1024 * 1024)

print(f"Conjunto de Treino:     {len(train_tokens):,} tokens ({train_mb:.2f} MB) -> {caminho_train}")
print(f"Conjunto de Validação:  {len(val_tokens):,} tokens ({val_mb:.2f} MB) -> {caminho_val}")

# ----------------------------------------------------
# 4. Teste de Decodificação e Amostra do Vocabulário
# ----------------------------------------------------
print("\nTeste de reconstrução de texto pelo tokenizador:")
amostra_original = "O Copom decidiu elevar a taxa Selic para conter as pressões inflacionárias."
ids_amostra = tokenizer.encode(amostra_original).ids
texto_reconstruido = tokenizer.decode(ids_amostra)

print(f"Original:      '{amostra_original}'")
print(f"Token IDs:     {ids_amostra}")
print(f"Reconstruído:  '{texto_reconstruido}'")

Treinando tokenizador ByteLevel BPE com vocab_size=4096...
Tokenizador salvo com sucesso em: /content/drive/MyDrive/llm_data/tokenizer

Codificando o corpus completo em tokens...
Total de caracteres de entrada: 14,102,171
Total de tokens gerados:        3,627,571
Taxa de compressão média:       3.89 caracteres/token
Conjunto de Treino:     3,264,813 tokens (6.23 MB) -> /content/drive/MyDrive/llm_data/train.bin
Conjunto de Validação:  362,758 tokens (0.69 MB) -> /content/drive/MyDrive/llm_data/val.bin

Teste de reconstrução de texto pelo tokenizador:
Original:      'O Copom decidiu elevar a taxa Selic para conter as pressões inflacionárias.'
Token IDs:     [49, 622, 2884, 3408, 261, 519, 1197, 323, 412, 284, 373, 1441, 2198, 16]
Reconstruído:  'O Copom decidiu elevar a taxa Selic para conter as pressões inflacionárias.'


In [5]:
import os
import math
import time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from tokenizers import ByteLevelBPETokenizer

# 1. Garante que a GPU está ativa
assert torch.cuda.is_available(), "GPU não detectada! Vá em Ambiente de execução -> Alterar tipo de ambiente de execução -> T4 GPU."
device = 'cuda'
print(f"Dispositivo de execução: {device} ({torch.cuda.get_device_name(0)})")

# 2. Caminhos no Google Drive
pasta_base = '/content/drive/MyDrive/llm_data'
caminho_train = os.path.join(pasta_base, 'train.bin')
caminho_val = os.path.join(pasta_base, 'val.bin')
pasta_tokenizer = os.path.join(pasta_base, 'tokenizer')
caminho_checkpoint = os.path.join(pasta_base, 'best_model.pt')

# 3. Hiperparâmetros Calibrados para GPU T4
batch_size = 32            # 32 sequências por passo
block_size = 256           # Janela de contexto
max_iters = 3000           # Total de iterações
eval_interval = 250        # Avalia a cada 250 iterações
eval_iters = 20            # Amostras rápidas para estimar loss
learning_rate = 5e-4
min_lr = 5e-5
warmup_iters = 100

n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
vocab_size = 4096

# 4. Carregamento de Lotes (Memmap rápido)
train_data = np.memmap(caminho_train, dtype=np.uint16, mode='r')
val_data = np.memmap(caminho_val, dtype=np.uint16, mode='r')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

# 5. Arquitetura Transformer Causal
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # FlashAttention nativo em CUDA
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=None,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP()

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(vocab_size, n_embd),
            wpe = nn.Embedding(block_size, n_embd),
            drop = nn.Dropout(dropout),
            h = nn.ModuleList([Block() for _ in range(n_layer)]),
            ln_f = nn.LayerNorm(n_embd),
        ))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            return self.lm_head(x[:, [-1], :]), None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.75, top_k=30):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# 6. Instanciação e Otimização
model = GPT().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo instanciado com {total_params / 1e6:.2f}M parâmetros.")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=1e-1)

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    if it > max_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# 7. Loop de Treino com Medição em Tempo Real
print("\nIniciando o treinamento na GPU...")
best_val_loss = float('inf')
t0 = time.time()

for iter_num in range(max_iters + 1):
    lr = get_lr(iter_num)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if iter_num % eval_interval == 0:
        losses = estimate_loss()
        tempo = time.time() - t0
        print(f"Passo {iter_num:>4}/{max_iters} | Treino Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f} | LR: {lr:.2e} | Tempo: {tempo:.1f}s")

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save({
                'model_state_dict': model.state_dict(),
                'val_loss': best_val_loss,
                'iter': iter_num,
                'config': {'n_embd': n_embd, 'n_head': n_head, 'n_layer': n_layer, 'block_size': block_size, 'vocab_size': vocab_size}
            }, caminho_checkpoint)

    if iter_num == max_iters:
        break

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print(f"\nTreino finalizado com sucesso! Melhor Val Loss: {best_val_loss:.4f}")
print(f"Modelo salvo em: {caminho_checkpoint}")

# 8. Teste de Inferência Imediata
print("\n" + "=" * 60)
print("TESTE DE GERAÇÃO TEXTUAL AUTOREGRESSIVA")
print("=" * 60)
tokenizer = ByteLevelBPETokenizer(
    os.path.join(pasta_tokenizer, "vocab.json"),
    os.path.join(pasta_tokenizer, "merges.txt")
)

prompts = [
    "O Copom decidiu elevar a taxa Selic",
    "No cenário de referência para a inflação",
    "A estabilidade do sistema financeiro nacional"
]

model.eval()
for p in prompts:
    ids = torch.tensor(tokenizer.encode(p).ids, dtype=torch.long, device=device).unsqueeze(0)
    saida = model.generate(ids, max_new_tokens=90, temperature=0.75, top_k=30)[0].tolist()
    print(f"\nPrompt: '{p}'")
    print(f"Continuação: {tokenizer.decode(saida)}\n" + "-" * 50)

Dispositivo de execução: cuda (Tesla T4)
Modelo instanciado com 12.30M parâmetros.

Iniciando o treinamento na GPU...
Passo    0/3000 | Treino Loss: 8.3761 | Val Loss: 8.4110 | LR: 0.00e+00 | Tempo: 13.1s
Passo  250/3000 | Treino Loss: 4.3113 | Val Loss: 5.5052 | LR: 4.97e-04 | Tempo: 63.4s
Passo  500/3000 | Treino Loss: 3.5318 | Val Loss: 5.0790 | LR: 4.79e-04 | Tempo: 116.3s
Passo  750/3000 | Treino Loss: 3.0976 | Val Loss: 4.8223 | LR: 4.46e-04 | Tempo: 170.1s
Passo 1000/3000 | Treino Loss: 2.8671 | Val Loss: 4.6738 | LR: 4.01e-04 | Tempo: 225.9s
Passo 1250/3000 | Treino Loss: 2.6121 | Val Loss: 4.5319 | LR: 3.47e-04 | Tempo: 283.0s
Passo 1500/3000 | Treino Loss: 2.5276 | Val Loss: 4.4357 | LR: 2.87e-04 | Tempo: 341.0s
Passo 1750/3000 | Treino Loss: 2.3968 | Val Loss: 4.3550 | LR: 2.27e-04 | Tempo: 399.6s
Passo 2000/3000 | Treino Loss: 2.2916 | Val Loss: 4.3865 | LR: 1.70e-04 | Tempo: 458.3s
Passo 2250/3000 | Treino Loss: 2.1909 | Val Loss: 4.3682 | LR: 1.20e-04 | Tempo: 516.8s
Pass

In [8]:
import os
import re
import random
import numpy as np
from tokenizers import ByteLevelBPETokenizer

pasta_base = '/content/drive/MyDrive/llm_data'
caminho_clean = os.path.join(pasta_base, 'input_clean.txt')
pasta_tokenizer = os.path.join(pasta_base, 'tokenizer')

# 1. Carrega o corpus limpo
with open(caminho_clean, 'r', encoding='utf-8') as f:
    texto = f.read()

# 2. Divide o texto em blocos/documentos preservando a integridade das seções
# Divide por quebras duplas de linha ou tags de seção
blocos = [b.strip() for b in re.split(r'\n{2,}', texto) if len(b.strip()) > 80]
print(f"Total de blocos/documentos identificados: {len(blocos):,}")

# 3. Embaralha com semente reprodutível
random.seed(42)
random.shuffle(blocos)

# 4. Reconstrói o texto intercalado
texto_shuffled = "\n\n".join(blocos)

# 5. Tokeniza com o BPE treinado
tokenizer = ByteLevelBPETokenizer(
    os.path.join(pasta_tokenizer, "vocab.json"),
    os.path.join(pasta_tokenizer, "merges.txt")
)

print("Tokenizando corpus unificado e balanceado...")
tokens = np.array(tokenizer.encode(texto_shuffled).ids, dtype=np.uint16)
num_tokens = len(tokens)

# 6. Divisão 90% Treino / 10% Validação
split_idx = int(num_tokens * 0.9)
train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

caminho_train = os.path.join(pasta_base, 'train.bin')
caminho_val = os.path.join(pasta_base, 'val.bin')

train_tokens.tofile(caminho_train)
val_tokens.tofile(caminho_val)

print("=" * 60)
print(f"Tokens de Treino (Balanceados):    {len(train_tokens):,} -> {caminho_train}")
print(f"Tokens de Validação (Balanceados): {len(val_tokens):,} -> {caminho_val}")
print("=" * 60)

Total de blocos/documentos identificados: 13,655
Tokenizando corpus unificado e balanceado...
Tokens de Treino (Balanceados):    3,242,348 -> /content/drive/MyDrive/llm_data/train.bin
Tokens de Validação (Balanceados): 360,261 -> /content/drive/MyDrive/llm_data/val.bin
